# Kaggle F1 Boosting Solution

This notebook contains a reproducible multi-class classification pipeline for the anonymized Kaggle challenge.

The workflow uses only `train_data.csv` for model selection through Stratified K-Fold cross-validation. The test set is used only at the end to generate the submission file. No target leakage, manual submission edits, or external labels are used.

## 1. Imports

This cell imports the required libraries for data handling, cross-validation, model training, evaluation, and saving artifacts.

In [ ]:
import itertools
import json
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

## 2. Configuration

These constants define the random seed, cross-validation setup, column names, and data paths. The path logic allows the notebook to run from either the project root or the `MLOPS` folder.

In [ ]:
RANDOM_STATE = 42
N_SPLITS = 5
ID_COL = "ID"
TARGET_COL = "target"
AVERAGE = "macro"

CURRENT_DIR = Path.cwd()
if (CURRENT_DIR / "train_data.csv").exists():
    DATA_DIR = CURRENT_DIR
elif (CURRENT_DIR.parent / "train_data.csv").exists():
    DATA_DIR = CURRENT_DIR.parent
else:
    DATA_DIR = CURRENT_DIR

ARTIFACT_DIR = Path("artifacts")
ARTIFACT_DIR.mkdir(exist_ok=True)

print(f"Using data directory: {DATA_DIR.resolve()}")

## 3. Evaluation Helper

This function trains one estimator with Stratified K-Fold cross-validation. It returns out-of-fold probabilities, averaged test probabilities, and fold-level F1 scores.

In [ ]:
def evaluate_model(name, estimator, X, y, X_test, folds):
    n_classes = len(np.unique(y))
    oof_proba = np.zeros((len(X), n_classes), dtype=float)
    test_probas = []
    fold_scores = []

    for fold, (tr_idx, va_idx) in enumerate(folds, start=1):
        model = clone(estimator)
        model.fit(X.iloc[tr_idx], y[tr_idx])

        va_proba = model.predict_proba(X.iloc[va_idx])
        test_proba = model.predict_proba(X_test)

        oof_proba[va_idx] = va_proba
        test_probas.append(test_proba)

        score = f1_score(y[va_idx], va_proba.argmax(axis=1), average=AVERAGE)
        fold_scores.append(score)
        print(f"{name} fold {fold}: {score:.6f}")

    oof_score = f1_score(y, oof_proba.argmax(axis=1), average=AVERAGE)
    return {
        "name": name,
        "oof_score": oof_score,
        "fold_mean": float(np.mean(fold_scores)),
        "fold_std": float(np.std(fold_scores, ddof=1)),
        "oof_proba": oof_proba,
        "test_proba": np.mean(test_probas, axis=0),
    }

## 4. Class Multiplier Tuning

Macro F1 can benefit from small class-level probability multipliers, especially with imbalanced classes. The multipliers are selected only from out-of-fold training predictions, not from the test set.

In [ ]:
def best_class_multipliers(y, oof_proba):
    grid = [0.65, 0.75, 0.85, 0.95, 1.0, 1.05, 1.15, 1.3, 1.5, 1.8, 2.2]
    best_score = f1_score(y, oof_proba.argmax(axis=1), average=AVERAGE)
    best_mult = np.ones(oof_proba.shape[1])

    for mult in itertools.product(grid, repeat=oof_proba.shape[1]):
        mult = np.asarray(mult, dtype=float)
        pred = (oof_proba * mult).argmax(axis=1)
        score = f1_score(y, pred, average=AVERAGE)
        if score > best_score:
            best_score = score
            best_mult = mult

    return best_score, best_mult

## 5. Load Data

The training file contains the target labels. The test file contains the same anonymized features without labels. The test set is not used for validation or model selection.

In [ ]:
train = pd.read_csv(DATA_DIR / "train_data.csv")
test = pd.read_csv(DATA_DIR / "test_data.csv")

feature_cols = [c for c in train.columns if c not in [ID_COL, TARGET_COL]]
X = train[feature_cols].copy()
X_test = test[feature_cols].copy()

encoder = LabelEncoder()
y = encoder.fit_transform(train[TARGET_COL].astype(str))
class_names = list(encoder.classes_)

print("Train shape:", train.shape)
print("Test shape:", test.shape)
print("Classes:", class_names)
print("Class distribution:")
print(train[TARGET_COL].value_counts())

## 6. Cross-Validation Splits

The validation strategy uses 5-fold Stratified K-Fold. Stratification preserves the class distribution in every fold.

In [ ]:
folds = list(StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE,
).split(X, y))

print(f"Number of folds: {len(folds)}")

## 7. Model Definitions

The candidate models use tree-based algorithms that work well on anonymized tabular data. Each model includes median imputation inside an sklearn pipeline.

In [ ]:
models = {
    "rf_600_leaf1_balanced": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestClassifier(
            n_estimators=600,
            min_samples_leaf=1,
            max_features="sqrt",
            class_weight="balanced_subsample",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )),
    ]),
    "extra_800_leaf1_balanced": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", ExtraTreesClassifier(
            n_estimators=800,
            min_samples_leaf=1,
            max_features="sqrt",
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )),
    ]),
    "xgb_balanced": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", XGBClassifier(
            objective="multi:softprob",
            eval_metric="mlogloss",
            num_class=len(class_names),
            n_estimators=450,
            max_depth=3,
            learning_rate=0.035,
            min_child_weight=2,
            subsample=0.85,
            colsample_bytree=0.85,
            reg_lambda=4.0,
            reg_alpha=0.2,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )),
    ]),
    "lgbm_balanced": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", LGBMClassifier(
            objective="multiclass",
            n_estimators=500,
            learning_rate=0.035,
            num_leaves=15,
            min_child_samples=12,
            subsample=0.85,
            colsample_bytree=0.85,
            reg_lambda=4.0,
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=-1,
            verbose=-1,
        )),
    ]),
    "cat_balanced": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", CatBoostClassifier(
            loss_function="MultiClass",
            iterations=500,
            depth=4,
            learning_rate=0.04,
            l2_leaf_reg=8.0,
            auto_class_weights="Balanced",
            random_seed=RANDOM_STATE,
            verbose=False,
            allow_writing_files=False,
        )),
    ]),
}

list(models.keys())

## 8. Train Individual Models

Each model is evaluated with the same cross-validation folds. The outputs are stored for both individual model comparison and ensemble construction.

In [ ]:
outputs = []

for name, model in models.items():
    print(f"
Running {name}")
    outputs.append(evaluate_model(name, model, X, y, X_test, folds))

## 9. Tune Individual Model Multipliers

This cell computes raw and tuned out-of-fold Macro F1 for every individual model.

In [ ]:
rows = []

for out in outputs:
    tuned_score, mult = best_class_multipliers(y, out["oof_proba"])
    out["tuned_score"] = tuned_score
    out["multipliers"] = mult
    rows.append({
        "model": out["name"],
        "oof_macro_f1": out["oof_score"],
        "tuned_oof_macro_f1": tuned_score,
        "fold_mean": out["fold_mean"],
        "fold_std": out["fold_std"],
        "multipliers": mult.tolist(),
    })

pd.DataFrame(rows).sort_values("tuned_oof_macro_f1", ascending=False)

## 10. Build and Evaluate Ensembles

The notebook evaluates all model combinations of size 2 to 5 by averaging predicted probabilities. The best ensemble is selected using out-of-fold Macro F1 only.

In [ ]:
base_outputs = list(outputs)

for combo_size in [2, 3, 4, 5]:
    for combo in itertools.combinations(base_outputs, combo_size):
        names = "+".join(o["name"] for o in combo)
        oof = np.mean([o["oof_proba"] for o in combo], axis=0)
        test_proba = np.mean([o["test_proba"] for o in combo], axis=0)

        score = f1_score(y, oof.argmax(axis=1), average=AVERAGE)
        tuned_score, mult = best_class_multipliers(y, oof)

        outputs.append({
            "name": f"ens_{names}",
            "oof_score": score,
            "tuned_score": tuned_score,
            "multipliers": mult,
            "oof_proba": oof,
            "test_proba": test_proba,
        })
        rows.append({
            "model": f"ens_{names}",
            "oof_macro_f1": score,
            "tuned_oof_macro_f1": tuned_score,
            "fold_mean": np.nan,
            "fold_std": np.nan,
            "multipliers": mult.tolist(),
        })

results = pd.DataFrame(rows).sort_values("tuned_oof_macro_f1", ascending=False)
results.head(15)

## 11. Save Results and Submission

The best model is selected by tuned out-of-fold Macro F1. The final submission is generated from the averaged test probabilities of the selected model or ensemble.

In [ ]:
best_name = results.iloc[0]["model"]
best = next(o for o in outputs if o["name"] == best_name)
multipliers = np.asarray(results.iloc[0]["multipliers"], dtype=float)

pred = (best["test_proba"] * multipliers).argmax(axis=1)
labels = encoder.inverse_transform(pred)

submission_path = f"submission_{best_name[:80].replace('+', '_')}_tuned_macro_f1.csv"
pd.DataFrame({ID_COL: test[ID_COL], TARGET_COL: labels}).to_csv(submission_path, index=False)

results.to_csv(ARTIFACT_DIR / "boosting_experiment_results.csv", index=False)
with open(ARTIFACT_DIR / "boosting_experiment_metadata.json", "w", encoding="utf-8") as f:
    json.dump({
        "best_model": best_name,
        "best_tuned_oof_macro_f1": float(results.iloc[0]["tuned_oof_macro_f1"]),
        "class_names": class_names,
        "feature_cols": feature_cols,
        "submission_path": submission_path,
    }, f, ensure_ascii=False, indent=2)

print(f"Best model: {best_name}")
print(f"Best tuned OOF Macro F1: {results.iloc[0]['tuned_oof_macro_f1']:.6f}")
print(f"Saved submission: {submission_path}")

## 12. Submission Preview

This preview confirms that the generated file follows the required Kaggle format: `ID,target`.

In [ ]:
pd.read_csv(submission_path).head()

## 13. MLOps: MLflow Tracking

The assignment requires tracking experiments with MLflow. The Python script `experiment_boosting_f1.py` contains the MLflow-integrated version of the same pipeline. It logs multiple runs, including individual models and the best ensemble.

Run the next cell when you want to reproduce the MLflow experiment database and artifacts.

In [ ]:
# This runs the MLflow-tracked Python script from the notebook.
# It creates mlflow.db, mlartifacts/, artifacts/, submissions/, and mlflow_experiment_summary.csv.
%run experiment_boosting_f1.py

## 14. MLflow Experiment Summary

After running the MLflow script, this cell displays the tracked runs and their validation metrics.

In [ ]:
mlflow_summary = pd.read_csv("mlflow_experiment_summary.csv")
mlflow_summary

## 15. Open the MLflow Dashboard

Run this command in PowerShell from the `MLOPS` folder, then open `http://127.0.0.1:5000` in your browser. Take a screenshot of the comparison table for the final report.

In [ ]:
print('cd "D:\ML_HM_FINISH_2\MLOPS"')
print('python -m mlflow ui --backend-store-uri sqlite:///D:/ML_HM_FINISH_2/MLOPS/mlflow.db --host 127.0.0.1 --port 5000')

## 16. MLOps Notes

- The tracked experiments use 5-fold Stratified Cross Validation.
- MLflow logs parameters, metrics, submissions, fold metrics, and the best model bundle.
- The best model bundle stores the trained estimators and preprocessing pipelines together.
- The test set is only used to generate the final submission, not for validation.